In [1]:
import os
os.environ["TORCH_CPP_LOG_LEVEL"] = "ERROR"

# Interactive plotting for ROI/table point selection cells
%matplotlib widget

from input_process import extract_frames
from table_detector import TableDetector
from utils.visualization import draw_bounces_on_split_table
import ball_tracking.transform as bt_transform
import torch
from PIL import Image
import numpy as np
import importlib
import notebook_helpers as nh

importlib.reload(bt_transform)
importlib.reload(nh)

CenterCropResizeFrame = bt_transform.CenterCropResizeFrame
read_json = nh.read_json
save_json = nh.save_json
read_image = nh.read_image
show_cropped_image = nh.show_cropped_image
save_table_points_preview = nh.save_table_points_preview
select_roi_jupyter = nh.select_roi_jupyter
get_roi_result = nh.get_roi_result
click_6_points_jupyter = nh.click_6_points_jupyter
update_split_draw_coordinates = nh.update_split_draw_coordinates
build_final_output_paths = nh.build_final_output_paths
convert_grouped_to_output = nh.convert_grouped_to_output
export_final_artifacts = nh.export_final_artifacts
reorganize_rallies_with_bounces_under_hits = nh.reorganize_rallies_with_bounces_under_hits
filter_based_on_score_changes = nh.filter_based_on_score_changes
summarize_rallies_counts = nh.summarize_rallies_counts
extract_frames_with_retry = nh.extract_frames_with_retry
prepare_video_inputs = nh.prepare_video_inputs
load_pipeline = nh.load_pipeline
detect_scoreboard_changes = nh.detect_scoreboard_changes
prepare_rallies = nh.prepare_rallies
generate_event_predictions = nh.generate_event_predictions
add_ball_tracking_to_rallies = nh.add_ball_tracking_to_rallies

CLASS_CONVERSION = {
    0: 'empty',
    1: 'far_table_bounce',
    2: 'far_table_forehand',
    3: 'far_table_backhand',
    4: 'far_table_serve',
    5: 'close_table_bounce',
    6: 'close_table_forehand',
    7: 'close_table_backhand',
    8: 'close_table_serve',
}

## Runtime Configuration
Set the video path, device, and model checkpoint folders for this run.

In [6]:
# Runtime configuration
window_size = 100
stride = 50
device = "cuda"
video_path = "../data/26WPF_AUS_W10_G_Yang_Qian_AUS v_Tian_TPE_game3.mp4"

from huggingface_hub import snapshot_download
ball_tracking_dir = snapshot_download("AugustRushG123/TOTNet")
event_detection_dir = snapshot_download("AugustRushG123/MFS_Model")

event_detection_model_folder = os.path.join(
    event_detection_dir,
    "E2E800MFS_SL_TTAV3(1.0)_FP16_8GPU_50epochs_lr1e-3_bs10_seed42"
)


ball_tracking_model_dir_path = os.path.join(
    ball_tracking_dir,
    "TOTNet_TTA_(5)_Bidirect_(512,512)_BallMask_50epochs_WBCE[1,2,3,3]_bs_ch32"
)

EVENT_DETECTION_THRESHOLD = 0.01
EVENT_WINDOWS = {
    "close_table_serve": 150,
    "far_table_serve": 150,
}

print(f"Device: {device}")
print(f"Video path: {video_path}")
print(f"Event model folder: {event_detection_model_folder}")
print(f"Ball tracking model folder: {ball_tracking_model_dir_path}")
print(f"Event detection threshold: {EVENT_DETECTION_THRESHOLD}")
print(f"Serve NMS windows: {EVENT_WINDOWS}")

## Model Loading
Load the event detection and ball tracking models, then build the transforms used later in the pipeline.

In [7]:
# Load models and transforms from configured paths
(
    event_detection_model,
    ball_tracking_model,
    event_transform,
    ball_transform,
    MEAN,
    STD,
) = load_pipeline(
    device=device,
    event_detection_model_folder=event_detection_model_folder,
    ball_tracking_model_dir_path=ball_tracking_model_dir_path,
 )

## Video Preparation
Extract frames, choose a sample frame for annotations, and create the output folders for this run.

In [8]:
video_inputs = prepare_video_inputs(
    video_path=video_path,
    device=device,
    extract_frames_fn=extract_frames,
    read_image_fn=read_image,
 )

frame_dir = video_inputs["frame_dir"]
fps_rate = video_inputs["fps_rate"]
game_name = video_inputs["game_name"]
image_files = video_inputs["image_files"]
sample_image_path = video_inputs["sample_image_path"]
sample_frame = video_inputs["sample_frame"]

In [ ]:
from datetime import datetime
from pathlib import Path

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path("outputs") / game_name / RUN_ID
VISUALS_DIR = OUTPUT_DIR / "visuals"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run output directory: {OUTPUT_DIR}")
print(f"Saved drawings folder: {VISUALS_DIR}")

## Scoreboard Setup
First select the close scoreboard region and save its preview. Then select the far scoreboard region in a separate step so both saved ROI images are different and usable for score detection.

In [ ]:
close_roi_coords = select_roi_jupyter(
    sample_frame,
    title="Select Close Scoreboard ROI",
)

In [ ]:
close_roi = get_roi_result(close_roi_coords)
print(close_roi)  # (x1, y1, x2, y2)
existing_close_coord = close_roi
show_cropped_image(
    sample_frame,
    existing_close_coord,
    save_path=str(VISUALS_DIR / "scoreboard_close_roi.png"),
)

In [ ]:
far_roi_coords = select_roi_jupyter(
    sample_frame,
    title="Select Far Scoreboard ROI",
)

### Far Scoreboard ROI
Select the far scoreboard region on the same sample frame. This preview is saved separately into the run `visuals` folder.

In [ ]:
far_roi = get_roi_result(far_roi_coords)
print(far_roi)  # (x1, y1, x2, y2)
existing_far_coord = far_roi

show_cropped_image(
    sample_frame,
    existing_far_coord,
    save_path=str(VISUALS_DIR / "scoreboard_far_roi.png"),
)

In [ ]:
scoreboard_result = detect_scoreboard_changes(
    frame_dir=frame_dir,
    fps_rate=fps_rate,
    device=device,
    existing_close_coord=existing_close_coord,
    existing_far_coord=existing_far_coord,
 )
changes = scoreboard_result["changes"]
final_score_close = scoreboard_result["final_score_close"]
final_score_far = scoreboard_result["final_score_far"]

In [ ]:
rally_result = prepare_rallies(
    frame_dir=frame_dir,
    window_size=window_size,
    stride=stride,
    event_transform=event_transform,
    ball_transform=ball_transform,
    event_detection_model=event_detection_model,
    device=device,
    fps_rate=fps_rate,
    class_conversion=CLASS_CONVERSION,
    scoreboard_changes=changes,
    threshold=EVENT_DETECTION_THRESHOLD,
    event_windows=EVENT_WINDOWS,
 )

dataset = rally_result["dataset"]
video_loader = rally_result["video_loader"]
pred_events = rally_result["pred_events"]
grouped_rallies = rally_result["grouped_rallies"]

## Table Mapping
Click the six table reference points, save the annotated selection, and generate the warped top-down table preview used for bounce mapping.

In [ ]:
img_trans = CenterCropResizeFrame(size=(512, 512), crop_ratio=(1, 1))
img = Image.open(sample_image_path).convert("RGB")
frame_np = np.array(img)  # PIL Image is RGB already
converted_img = img_trans(frame_np)
state = click_6_points_jupyter(converted_img)

In [ ]:
table_points = state["points"]
save_table_points_preview(
    converted_img,
    table_points,
    str(VISUALS_DIR / "table_points_selection.png"),
)
table_detector = TableDetector(image_path=sample_image_path, topdown_width=1525, topdown_height=2740)
table_detector.compute_homographies(corners6=table_points)
table_detector.warp_table(save_path=str(VISUALS_DIR / "table_topdown.png"))

In [ ]:
grouped_rallies = add_ball_tracking_to_rallies(
    grouped_rallies=grouped_rallies,
    dataset=dataset,
    ball_tracking_model=ball_tracking_model,
    table_detector=table_detector,
    device=device,
)
print(f"Ball tracking complete for {len(grouped_rallies)} rallies (kept in memory).")

## Ball Tracking And Visualization
Attach bounce locations to rallies, save the split-table drawing, then review the rally summary before export.

In [ ]:
BOUNCE_DRAWING_PATH = VISUALS_DIR / "bounces_split.png"

draw_bounces_on_split_table(
    grouped_rallies,
    save_path=str(BOUNCE_DRAWING_PATH),
)
print(f"Saved bounce drawing: {BOUNCE_DRAWING_PATH}")

In [ ]:
# Summarize events per rally in memory only (no file output)

SPLIT_W, SPLIT_L = 153.0, 137.0
TABLE_W, TABLE_H = 1525.0, 2740.0

summaries, invalid_bounce_total, bounce_total = summarize_rallies_counts(
    grouped_rallies, SPLIT_W, SPLIT_L, TABLE_W, TABLE_H
)

for s in summaries:
    print(f"Rally {s['rally_id']}: bounces(close/far)={s['close_bounce']}/{s['far_bounce']}, "
          f"forehand={s['total_forehand']}, backhand={s['total_backhand']}, invalid_bounces={s['invalid_bounces']}")
print(f"Invalid bounces: {invalid_bounce_total}/{bounce_total}")
print("Rally event counts kept in memory only.")

In [ ]:
GAME_LABEL = "Game 3"
CLOSE_PLAYER = "Qian"
FAR_PLAYER = "Tian"
FPS = fps_rate
INCLUDE_BOUNCES_IN_OUTPUT = True

print("Export configuration ready.")
print(f"Game label: {GAME_LABEL}")
print(f"Close player: {CLOSE_PLAYER}")
print(f"Far player: {FAR_PLAYER}")

## Export Artifacts
Write the final JSON, XML, score timeline, and keep all saved drawings under the same run folder for later review.

In [ ]:
OUTPUT_PATHS = build_final_output_paths(game_name, output_dir=str(OUTPUT_DIR))
SCORE_TIMELINE_PATH = OUTPUT_DIR / f"{game_name}_score_timeline.json"

legacy_paths = [
    f"grouped_rallies_{game_name}.json",
    f"grouped_rallies_with_nesting_{game_name}.json",
    f"grouped_rallies_with_nesting_{game_name}_enriched.json",
    f"rally_event_counts_{game_name}.json",
    "bounces_on_table.jpg",
    "bounces_on_table_split.png",
]

export_result = export_final_artifacts(
    grouped_rallies=grouped_rallies,
    game_name=game_name,
    game_label=GAME_LABEL,
    close_player=CLOSE_PLAYER,
    far_player=FAR_PLAYER,
    fps=FPS,
    output_dir=str(OUTPUT_DIR),
    summary_path=OUTPUT_PATHS["summary_path"],
    xml_ready_json_path=OUTPUT_PATHS["xml_ready_json_path"],
    xml_path=OUTPUT_PATHS["xml_path"],
    cleanup_paths=legacy_paths,
    table_w=TABLE_W,
    table_h=TABLE_H,
    split_w=SPLIT_W,
    split_l=SPLIT_L,
    include_bounces_in_output=INCLUDE_BOUNCES_IN_OUTPUT,
 )

save_json(str(SCORE_TIMELINE_PATH), changes)

print(f"Run output directory: {OUTPUT_DIR}")
print(f"Saved drawings folder: {VISUALS_DIR}")
print(f"Saved final rally summary: {export_result['summary_path']}")
print(f"Saved XML-ready JSON: {export_result['xml_ready_json_path']}")
print(f"Saved XML: {export_result['xml_path']}")
print(f"Saved score timeline: {SCORE_TIMELINE_PATH}")
print(f"Updated bounce split coords: {export_result['updated_bounces']}")
print(f"Total output instances: {export_result['instance_count']}")
if export_result['removed_files']:
    print("Removed temporary files:")
    for path in export_result['removed_files']:
        print(f"  - {path}")

In [ ]:
print("Split-coordinate enrichment is handled inside the previous export cell.")
print("No additional files are written here.")
print("XML-ready JSON export is handled inside the previous export cell.")
print("No additional files are written here.")
print("Final artifacts are created in this run folder:")
print(f"  - {OUTPUT_PATHS['summary_path']}")
print(f"  - {OUTPUT_PATHS['xml_ready_json_path']}")
print(f"  - {OUTPUT_PATHS['xml_path']}")
print(f"  - {SCORE_TIMELINE_PATH}")
print(f"Run directory: {OUTPUT_DIR}")